In [174]:
class Parser:
    def __init__(self, input):
        self.input = input
        self.head = 0
        self.length = len(input)

    def peek(self):
        return self.input[self.head]

    def next(self):
        ret = self.peek()
        self.head += 1
        return ret

    def r(self):
        self.head = 0

In [175]:
class PRV():
    def __init__(self,v=[],s=True,e=''):
        self.s = s
        self.v = v
        self.e = e
        
    def __bool__(self): return self.s
    
    def __repr__(self): return f"({self.s},{self.v},{self.e})"
    
    def __eq__(self,other): return self.v==other.v and self.s==other.s and self.e==other.e

In [176]:
def char():
    def p(input):
        try:
            n = input.next()
            return PRV([n])
        except:
            return PRV(s=False,e='end of input')

    return p

In [177]:
assert char()(Parser("1")) == PRV(['1'])

In [178]:
def satisfy(parser, acceptor):
    def p(input):
        head = input.head
        res = parser(input)
        if res:
            if acceptor(res.v): return res
            else: 
                input.head = head
                return PRV(s=False,e='satisfy failed')
        else:
            return res

    return p

In [179]:
def digit():
    return satisfy(char(), lambda x: x[0] in "0123456789")


def ascii_letter():
    return satisfy(
        char(), lambda x: x[0] in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
    )

In [180]:
assert digit()(Parser("1")) == PRV(['1'])
assert digit()(Parser("a")) == PRV(s=False,e='satisfy failed')

In [181]:
assert ascii_letter()(Parser("1")) == PRV(s=False,e='satisfy failed')
assert ascii_letter()(Parser("a")) == PRV(['a'])

In [182]:
def many(parser):
    def p(input):
        acc = []
        head = input.head
        while res := parser(input):
            head = input.head
            acc.append(res.v)
        input.head=head
        return PRV(acc)

    return p

In [183]:
assert many(digit())(Parser("123abc")) == PRV([['1'],['2'],['3']])
assert many(digit())(Parser("abc")) == PRV([])

In [184]:
def sequence(parsers):
    def p(input):
        head =input.head
        acc = []
        for parser in parsers:
            res = parser(input)
            if res: acc.append(res.v)
            else: 
                input.head=head
                return PRV(s=False,e=f"sequence failed")
            
        
        return PRV([x for x in acc if x])
    
    return p

In [185]:
assert sequence([digit(),digit(),digit()])(Parser("123")) == PRV([['1'],['2'],['3']])
assert sequence([digit(),ascii_letter(),digit()])(Parser("1a3")) == PRV([['1'],['a'],['3']])

In [186]:
def mapper(parser, funcs):
    def p(input):
        res = parser(input)
        if res:
            for f in funcs:
                res.v = f(res.v)
        return res

    return p

In [187]:
def accumulator(parser, acc):
    def p(input):
        while True:
            h = input.head
            try:
                acc.add(parser(input))
            except:
                input.head = h
                break
    return p

In [188]:
def digits():
    return mapper(sequence([digit(), many(digit())]), 
                  [lambda x: [''.join([y for xs in [x[0]]+x[1] for y in xs])]])

In [189]:
assert digits()(Parser("123")) == PRV(['123'])
assert digits()(Parser("abc")) == PRV(s=False,e='sequence failed')

In [190]:
def integer():
    return mapper(digits(), [lambda x: [int(x[0])]])

In [191]:
assert integer()(Parser("123")) == PRV([123])

In [192]:
def ws():
    return many(satisfy(char(), lambda x: x[0] in " \t"))

def _ws():
    def p(input):
        res = ws()(input)
        res.v = []
        return res
    return p
    

In [193]:
def nl():
    return satisfy(char(), lambda x: x == "\n")

def _nl():
    def p(input):
        res = _nl()(input)
        res.v = []
        return res
    return p
